# Lab 07 (Local): A Forensic Agent Over an Android Image

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs on:** your own computer, with a local Qwen model via **Ollama**. Nothing leaves your machine.

> This is the **local** version of Lab 07. It is the same lab as the Colab version, except the
> AI model (a small free model called Qwen) runs on your own PC through
> [Ollama](https://ollama.com), a free program that runs AI models on your own computer.
> Because the model is on your machine, open this notebook in **Jupyter on your computer**.
> Colab will not work here: it runs on Google's computers and cannot reach a model on yours.

---

**The investigation problem.** To answer a question like *which accounts are on this phone?*, an
examiner normally has to know which file on the phone holds the answer, open it, and dig the
answer out. In this lab you build a small **agent**: an AI assistant that is allowed to open
files itself, one at a time, and must tell you **which file its answer came from**. You ask in
plain English; it does the searching; you check its work.

You build the agent **by hand**, in three pieces:

1. **Tools:** a few safe, read-only actions the AI may ask for (list a folder, read a file, look
   inside a database, find a file by name, search for a word).
2. **A loop:** the AI thinks, picks one tool, sees what the tool found, and repeats.
3. **A stop rule:** when it has an answer, it must **cite the file**, and the notebook checks
   that it really opened that file.

**How this notebook flows** (each step hands its result to the next):

- **Part 0:** set up the AI model and build a small practice phone image.
- **Part A:** build the tools, the rulebook, and the loop, then ask a first question.
- **Part B:** ask the same question with and without the agent, and compare.
- **Part C:** ask your own questions.
- **Optional:** point the agent at a real public phone image, or let it write its own code.
- **Write-up and Submit.**

## Part 0: One-Time Setup

Before any investigation, you need two things: a working connection to an AI model, and some
evidence to examine. Part 0 gives you both. Run these cells once, top to bottom.

### Step 0.1: Install Ollama, download a Qwen model, and install the notebook's helper

**Purpose:** put a free AI model on your own computer, so the investigation never sends
evidence over the internet.

**Input:** nothing from earlier steps. You need an internet connection only for these one-time
downloads.

**How it works:** first, do this **once** in a normal terminal on your own machine (not in this
notebook):

1. Install Ollama from **https://ollama.com/download** (Windows, macOS, or Linux).
2. Download a small Qwen model:
   ```
   ollama pull qwen3:8b
   ```
3. Make sure Ollama is running. When it runs, it listens on your own computer at
   `http://localhost:11434` ("localhost" means "this computer"). On most installs it starts
   by itself; if not, run `ollama serve` in a terminal.

Then run the cell below. It installs `openai`, a small Python helper for sending messages to an
AI model. Ollama understands the same kind of messages, so this one helper works with it.

**Output:** a few install messages, or nothing at all. That is normal. Step 0.2 uses the helper.

In [ ]:
# This cell installs one helper library (ready-made code) the notebook needs.
# openai: a small helper for sending messages to an AI model and reading its replies.
#         Ollama on your PC understands the SAME kind of messages, so we point this
#         helper at your own computer instead of a cloud service.
%pip install -q openai

### Step 0.2: Connect to your local Qwen model

**Purpose:** check that the notebook can talk to the AI model running on your own computer.
No key, no internet, and no evidence leaves your machine.

**Input:** the `openai` helper from Step 0.1, and Ollama running with the `qwen3:8b` model you
downloaded.

**How it works:** the cell creates a **client** (the connection the notebook uses to send
messages to the model) pointed at Ollama on your computer. It then sends one tiny test
message, "Reply with one word: ready", to prove the model answers.

**Output:** `Model OK: qwen3:8b -> ...` when it works, or a checklist if Ollama cannot be
reached. This step gives you `client` and `MODEL`, which every AI step below uses.

In [ ]:
# This cell connects the notebook to the AI model running on YOUR computer (through Ollama).
from openai import OpenAI  # the helper from Step 0.1; Ollama understands its messages

# Where to find the model: 'localhost' means this computer, 11434 is the door Ollama listens on.
BASE_URL = 'http://localhost:11434/v1'  # your local Ollama server
MODEL = 'qwen3:8b'  # the model you downloaded. For a smaller, faster one try 'qwen3:4b'.

# The client is the notebook's line to the model. Ollama needs no real key, so any word will do.
client = OpenAI(base_url=BASE_URL, api_key='ollama')  # Ollama ignores the key

# Send one tiny test message. try/except means: if this fails, print help instead of crashing.
try:
    probe = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip()[:40])
except Exception as err:
    print('Could not reach Ollama ->', err)
    print('\nChecklist: is Ollama installed and running? Did you run  ollama pull qwen3:8b ?')

### Step 0.3: Build a small practice Android phone image

**Purpose:** give the agent some evidence to investigate. A real phone extraction (a full copy of
a phone's files) is many gigabytes, so we build a tiny, made-up stand-in that is laid out the same
way. Everything in it is fictional; the phone's owner, "Nadia Okafor", is not a real person.

**Input:** nothing from earlier steps. Every file's contents are written inside the cell below.

**How it works:** the cell creates a folder called `android_image` and fills it with files at the
same paths a real Android phone uses. Some are plain text files (the phone's model details, the list
of installed apps, saved Wi-Fi networks, a WhatsApp profile). Others are **SQLite databases**: small
database files, which is how most Android apps store their data (Gmail accounts, contacts, text
messages, WhatsApp messages).

**Output:** a list of every file in the practice image. This folder, saved as `ROOT`, is the
evidence the agent searches from Step A.1 onward.

In [ ]:
# This cell builds a tiny, made-up Android phone image on disk for the agent to investigate.
import sqlite3   # sqlite3: Python's built-in reader/writer for SQLite files (how Android stores app data)
import os, json
from pathlib import Path  # Path: a readable way to refer to files and folders

# ROOT is the top folder of the practice image. Every tool later on stays inside it.
ROOT = Path('android_image')

# The plain text files: a dictionary (a list of name -> value pairs) of file path -> file contents.
TEXT_FILES = {
    'android_image/build.prop': '# Synthetic build.prop (teaching data, fictional)\nro.product.manufacturer=Google\nro.product.brand=google\nro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\nro.build.version.release=14\nro.build.version.sdk=34\nro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n',
    'android_image/data/system/packages.list': 'com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\ncom.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\ncom.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\ncom.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\ncom.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\norg.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n',
    'android_image/data/system/users/0/settings_secure.xml': '<?xml version=\'1.0\' encoding=\'utf-8\' standalone=\'yes\' ?>\n<settings version="241">\n  <setting id="1" name="android_id" value="a1b2c3d4e5f60718" package="android" />\n  <setting id="2" name="bluetooth_name" value="Pixel 7a" package="com.android.settings" />\n  <setting id="3" name="device_name" value="Nadia\'s Pixel 7a" package="com.android.settings" />\n</settings>\n',
    'android_image/data/misc/wifi/WifiConfigStore.xml': '<?xml version=\'1.0\' encoding=\'utf-8\' standalone=\'yes\' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n    <Network><string name="SSID">&quot;HomeNet_5G&quot;</string><long name="LastConnectedTime" value="1721295600000" /></Network>\n    <Network><string name="SSID">&quot;BrightBeans Cafe&quot;</string><long name="LastConnectedTime" value="1720950000000" /></Network>\n    <Network><string name="SSID">&quot;Okafor-Office&quot;</string><long name="LastConnectedTime" value="1721210400000" /></Network>\n  </NetworkList>\n</WifiConfigStoreData>\n',
    'android_image/data/data/com.whatsapp/files/me.json': '{\n  "jid": "15550143788@s.whatsapp.net",\n  "pushname": "Nadia O",\n  "registered": true,\n  "last_backup": "2024-07-18"\n}',
}
# A loop: for each file in the dictionary, create its folders, then write the file.
for rel, content in TEXT_FILES.items():
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)

# A function (a reusable recipe) that creates one SQLite database file:
# make the folders, start from a fresh file, create the table(s), then add the rows.
def make_db(rel, schema, inserts):
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists(): p.unlink()
    con = sqlite3.connect(p); con.executescript(schema)
    for sql, params in inserts: con.executemany(sql, params)
    con.commit(); con.close()

# The four app databases. Each one is: the file path, the table layout, and the rows to add.
# Times are Unix times (seconds, or milliseconds for messages, counted from 1 January 1970).
make_db('android_image/data/data/com.google.android.gm/databases/accounts.db',
        'CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);',
        [('INSERT INTO accounts VALUES (?,?,?)', [
            ('nadia.okafor@gmail.com', 'com.google', 1721295600),
            ('+15550143788', 'com.whatsapp', 1721240000)])])
make_db('android_image/data/data/com.android.providers.contacts/databases/contacts2.db',
        'CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);',
        [('INSERT INTO contacts VALUES (?,?,?)', [
            ('Marcus Webb', '+15550142210', 31), ('Priya Shah', '+15550148890', 12),
            ('Dad', '+15550149001', 54), ('BrightBeans Cafe', '+15550140077', 3),
            ('Alina K (work)', '+15550147325', 20)])])
make_db('android_image/data/data/com.android.providers.telephony/databases/mmssms.db',
        'CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);',
        [('INSERT INTO sms VALUES (?,?,?,?)', [
            ('+15550142210', 1721200000000, 'Are we still on for Friday?', 1),
            ('+15550142210', 1721200600000, 'Yes, 6pm at the usual place.', 2),
            ('+15550149001', 1721240500000, 'Landed safe, call you tonight.', 2),
            ('+15550148890', 1721250000000, 'Can you send the invoice?', 1),
            ('+15550149001', 1721299000000, 'Thanks Dad, talk soon.', 2)])])
make_db('android_image/data/data/com.whatsapp/databases/msgstore.db',
        'CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);',
        [('INSERT INTO messages VALUES (?,?,?,?)', [
            ('Marcus Webb', 1721260000000, 0, 'Did the package arrive?'),
            ('Marcus Webb', 1721260400000, 1, 'Not yet, will check tomorrow.'),
            ('Priya Shah', 1721280000000, 1, 'Sent the files to your email.')])])

# Print every file in the image, so you can see what the agent will be able to explore.
print('Built synthetic image under', ROOT.resolve(), '\n')
for dp, dn, fn in sorted(os.walk(ROOT)):
    for f in sorted(fn): print('  ', os.path.join(dp, f))

## Part A: Build the Agent by Hand

An AI model on its own can only answer from what it already "remembers". It has never seen this
phone, so on its own it can only guess. An **agent** is an AI model that is allowed to *act*: it
asks for a tool, looks at what the tool found, and decides what to do next, the way an examiner
opens one file, reads it, and then decides which file to open next.

You build the agent in three steps: the **tools** (Step A.1), the **rulebook** the AI follows
(Step A.2), and the **loop** that runs the investigation (Step A.3).

### Step A.1: Give the agent its read-only tools

**Purpose:** decide exactly what the agent is allowed to do to the evidence. The AI cannot touch
files directly. It can only ask for one of the tools we write here, and every tool is
**read-only** (it can look, never change) and **locked inside the image folder** (it cannot
wander off to other files on the computer).

**Input:** the practice phone image `ROOT` from Step 0.3.

**How it works:** the cell writes five small tools, each one a short piece of Python:

- `list_dir(path)`: list what is inside a folder.
- `read_file(path)`: show the text inside a file. For a database it does not dump raw bytes; it
  lists the database's tables and says to use `run_sql` instead.
- `run_sql(path, query)`: ask a database for rows. **SQL** is a short language for asking a database
  questions, such as `SELECT name FROM accounts`. Only questions that read (`SELECT`) are allowed.
- `find_files(name)`: find files anywhere in the image whose path contains a word.
- `search_text(keyword)`: search inside the text files for a word, to learn which file to open next.

Good forensic practice is built in: the database tools work on a **temporary copy** of the
database, never the evidence file itself, because even opening a database can quietly create new
files beside it.

**Output:** a quick self-test of three tools on the image. The tools are collected in a dictionary
called `TOOLS` (tool name -> tool), which the loop in Step A.3 uses to run whichever tool the AI asks
for.

In [ ]:
# This cell writes the small, safe toolbox the agent may use. Nothing here can change the image.
# Skim it to see what each tool does; you do not need to follow every line.
import sqlite3, os, json, shutil, tempfile
from pathlib import Path

# File endings we treat as readable text (everything else may be binary data).
TEXT_EXT = {'.txt', '.xml', '.json', '.prop', '.list', '.log', '.csv', '.ini', '.conf'}

# A safety helper used by every tool: the AI hands us a path as text, and we refuse it
# if it points anywhere outside the image folder.
def _resolve(path):
    # Turn a path from the model into a real path, and refuse anything outside the image folder.
    base = ROOT.resolve()
    path = (path or '.').strip().lstrip('/')
    target = base if path in ('', '.') else (ROOT / path).resolve()
    if target != base and base not in target.parents:
        raise ValueError('path is outside the image')
    return target

# Tool 1: list_dir. Like opening a folder in a file browser.
def list_dir(path='.'):
    # List the entries in a folder (folders shown with a trailing '/').
    t = _resolve(path)
    if not t.exists():  return f'No such path: {path}'
    if t.is_file():     return f'{path} is a file, not a folder. Use read_file or run_sql.'
    items = [e + ('/' if (t / e).is_dir() else '') for e in sorted(os.listdir(t))]
    return '\n'.join(items) if items else '(empty folder)'

# A helper for the database tools (not a tool itself): it copies the database first,
# so we only ever open the copy and the evidence file stays untouched.
def open_db_copy(t):
    # Copy the database, plus its '-wal' side file (which can hold the newest rows), to a
    # temporary folder. Even opening the original read-only can add files beside it, so we
    # never open the evidence file itself.
    tmp = Path(tempfile.mkdtemp()); shutil.copy2(t, tmp / 'evidence.db')
    if Path(str(t) + '-wal').exists(): shutil.copy2(str(t) + '-wal', tmp / 'evidence.db-wal')
    return sqlite3.connect(tmp / 'evidence.db')

# Tool 2: read_file. Shows up to 1,500 characters of a file, so the AI gets a short sample.
def read_file(path, max_chars=1500):
    # Show a text file. Pretty-prints JSON; for a SQLite database, points you to run_sql.
    t = _resolve(path)
    if not t.exists() or t.is_dir():  return f'No such file: {path}'
    with open(t, 'rb') as f: head = f.read(16)   # just the first bytes, even for a huge file
    if head[:15] == b'SQLite format 3':
        con = open_db_copy(t)
        tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
        con.close()
        return f'[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]'
    if t.suffix.lower() not in TEXT_EXT and b'\x00' in head:
        return f'[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): {head.hex(" ")}.]'
    text = t.read_text(errors='replace')
    if t.suffix.lower() == '.json':
        try: text = json.dumps(json.loads(text), indent=2)
        except Exception: pass
    return text[:max_chars] + ('' if len(text) <= max_chars else '\n...[truncated]')

# Tool 3: run_sql. Asks a database for rows. Only reading questions are allowed, and at
# most 50 rows come back.
def run_sql(path, query):
    # Run ONE read-only SELECT against a SQLite database and return the rows.
    t = _resolve(path)
    if not t.exists():  return f'No such database: {path}'
    q = (query or '').strip().rstrip(';')
    if not q.lower().startswith(('select', 'pragma')):
        return 'Only read-only SELECT queries are allowed.'
    try:
        con = open_db_copy(t)   # a temporary copy: the evidence file itself is never opened
        cur = con.execute(q); cols = [c[0] for c in cur.description] if cur.description else []
        rows = cur.fetchmany(50); con.close()
    except Exception as e:
        return f'SQL error: {e}'
    if not rows: return '(no rows)'
    return '\n'.join([' | '.join(cols)] + [' | '.join(str(c) for c in r) for r in rows])

# Tool 4: find_files. Finds files by (part of) their name, anywhere in the image.
def find_files(name):
    # Find files anywhere in the image whose path contains `name` (case-insensitive).
    base = ROOT.resolve()
    hits = [str(p.relative_to(base)) for p in sorted(base.rglob('*'))
            if p.is_file() and (name or '').lower() in str(p.relative_to(base)).lower()]
    return '\n'.join(hits[:50]) if hits else f'No files matched {name!r}.'

# Tool 5: search_text. Looks for a word inside the text files (databases are skipped).
def search_text(keyword, max_hits=20):
    # Search inside readable text files for a keyword; returns path:line for each match.
    base = ROOT.resolve(); out = []
    for p in sorted(base.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in TEXT_EXT: continue
        for i, line in enumerate(p.read_text(errors='replace').splitlines(), 1):
            if (keyword or '').lower() in line.lower():
                out.append(f'{p.relative_to(base)}:{i}: {line.strip()[:120]}')
                if len(out) >= max_hits: return '\n'.join(out)
    return '\n'.join(out) if out else f'No text matches for {keyword!r}.'

# The toolbox: tool name -> tool. The loop in Step A.3 looks up the AI's chosen tool here.
TOOLS = {'list_dir': list_dir, 'read_file': read_file, 'run_sql': run_sql,
         'find_files': find_files, 'search_text': search_text}

# A quick self-test: run three tools ourselves and print what they return.
print('list_dir("data/data") ->\n' + list_dir('data/data'))
print('\nfind_files("accounts") ->\n' + find_files('accounts'))
print('\nsearch_text("android_id") ->\n' + search_text('android_id'))

### Step A.2: Write the agent's rulebook, and a reader for its replies

**Purpose:** tell the AI what its job is and how to reply, so that the notebook can understand
which tool it wants. Without it, the AI would reply in ordinary sentences the code cannot act on.

**Input:** the five tool names from Step A.1, which the rulebook lists for the AI.

**How it works:** the cell has two parts.

1. `SYSTEM_PROMPT`: the **prompt** (the instructions we give the AI) that it reads before every
   question. It gives the AI a name (Sectra), its job (help a forensic examiner with an Android
   extraction), the list of tools, and strict rules: open files before claiming anything, never
   invent data, and always cite the file used. It also says the reply must be a short **structured
   reply** in a format called JSON (simple `"name": value` pairs in curly brackets), either
   "use this tool" or "here is my final answer and the file it came from".
2. `parse_action`: a small reader that finds that structured reply in the AI's text, even if the
   AI wraps it in extra words, and turns it into something Python can use.

**Output:** nothing is printed. This step gives Step A.3 the rulebook `SYSTEM_PROMPT` and the
reader `parse_action`.

In [ ]:
# This cell writes the rulebook the AI follows, plus a reader for its structured replies.
import json, re

# The rulebook (the 'system prompt'). The text between the ''' marks is sent to the AI, word
# for word, at the start of every investigation. Read it: it is how we steer the AI.
# The '/no_think' at the very end asks Qwen models to skip their long private 'thinking'.
SYSTEM_PROMPT = '''You are Sectra, a digital-forensics examiner's assistant. You are investigating an
Android device extraction (a copy of a phone's files) to answer the examiner's question. You cannot
see any file until you open it with a tool, and every answer must be based ONLY on what the tools return.

TOOLS (use exactly one per step):
- list_dir(path): list the entries in a folder. Use "." for the top of the image.
- read_file(path): show the text contents of a file.
- run_sql(path, query): run ONE read-only SQL SELECT against a SQLite database and return rows.
- find_files(name): list files anywhere in the image whose path contains name.
- search_text(keyword): search inside text files for a keyword; returns path:line matches.

Reply with ONE JSON object and nothing else.
To use a tool:
  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}
When you can answer:
  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}

RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, or data;
if a tool errors, try another path or tool; if the evidence is not there, say so plainly; always cite
the exact file in evidence_path. Keep thoughts to one sentence. /no_think'''

# The reader. It returns the AI's reply as a dictionary (name -> value pairs), or None if
# the reply did not contain a readable structured reply.
def parse_action(raw):
    # Pull the JSON object out of the model's reply, tolerating <think> blocks and ``` code fences.
    # (Some models write their private 'thinking' between <think> marks first; we drop it.)
    text = re.sub(r'<think>.*?</think>', '', raw, flags=re.S)
    text = text.replace('```json', '').replace('```', '')
    # Find the first '{', then count brackets until the matching '}' closes it.
    start = text.find('{')
    if start == -1: return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                try: return json.loads(text[start:i + 1])
                except Exception: return None
    return None

### Step A.3: Run the agent loop, with an evidence check

**Purpose:** put the pieces together into a working agent, and ask it a first real question:
*Which email accounts are signed in on this device?*

**Input:** the connection `client` and `MODEL` from Step 0.2, the toolbox `TOOLS` from Step A.1,
and the rulebook `SYSTEM_PROMPT` and reader `parse_action` from Step A.2.

**How it works:** `run_agent` repeats one simple cycle, up to 12 times: **think, act, observe**.

1. **Think:** send the AI the rulebook, the question, and everything found so far, and ask for its
   next move.
2. **Act:** if it asks for a tool, run that one tool on the image.
3. **Observe:** add what the tool found to the conversation, and go round again.

This think-act-observe cycle is often called the **ReAct** pattern (short for "reason and act").

Before it accepts a final answer, the loop runs an **evidence check**. Read-only tools limit what
the AI can *look at*, but not what it can *say*: it could still name a file it never opened. So
the notebook, not the AI, checks the answer. If the agent has not asked for any tool yet, or the
answer cites no file, or the cited file never appeared in the agent's tool requests or in what
the tools returned, the answer is **sent back** with a note telling the AI to open the right file
first. This loop sends an answer back at most twice; after that it accepts the answer as it is.
So the check helps, but **you** are still the examiner: read the trace and confirm the file
yourself.

**Output:** a step-by-step trace (each `thought`, the `action` the AI chose, and a short `result`
from the tool), then `ANSWER:` and `EVIDENCE:` (the file it cited). If the check sends an answer
back, you will see a line `evidence check: answer sent back`. The final reply is saved as
`result`. Part B reuses `run_agent` for a comparison.

In [ ]:
# This cell writes the agent loop and runs it once. It drives the AI think -> act -> observe
# until it can answer.
def run_agent(question, max_steps=12, verbose=True):
    # The conversation so far, as a list of messages: first the rulebook, then the question.
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Investigate and answer this question: {question}'}]
    seen = ''          # a running record of every tool request and everything the tools returned
    checks = 0         # how many times the evidence check has sent an answer back
    # A loop: one pass per step, at most max_steps passes.
    for step in range(1, max_steps + 1):
        # THINK: send the whole conversation to the AI and get its next move as text.
        # temperature=0 asks the AI to be as consistent as possible, not creative.
        raw = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0).choices[0].message.content
        messages.append({'role': 'assistant', 'content': raw})
        move = parse_action(raw)
        # If we could not read a structured reply, ask the AI to try again in the right format.
        if move is None:
            messages.append({'role': 'user', 'content': 'That was not valid JSON. Reply with ONE JSON object only.'})
            continue
        if verbose and move.get('thought'): print(f'Step {step} | thought: {move["thought"]}')
        # Did the AI give a final answer? Then check it before accepting it.
        if 'final_answer' in move:
            # EVIDENCE CHECK: our code (not the AI) decides if the answer is backed by evidence.
            # The agent must have asked for a tool, it must cite a file, and that file must appear in
            # the record `seen`. If not, send the answer back (at most twice), with a note.
            cited = move.get('evidence_path') or ''
            if checks < 2 and (not seen or not cited or cited not in seen):
                checks += 1
                problem = ('You have not opened that file with a tool. Open the file that holds the '
                           'evidence, then answer only from what it shows, citing that file.')
                if verbose: print('        evidence check: answer sent back (cited file was never opened)')
                messages.append({'role': 'user', 'content': problem})
                continue
            if verbose:
                print(f'\nANSWER: {move["final_answer"]}')
                print(f'EVIDENCE: {move.get("evidence_path", "(none given)")}')
            return move
        # ACT: the AI asked for a tool. Look it up in TOOLS and run it with the AI's inputs.
        # try/except: if the tool fails, report the error to the AI instead of crashing.
        tool, args = move.get('action'), move.get('action_input', {}) or {}
        if tool not in TOOLS:
            observation = f'Unknown tool {tool!r}. Use one of: ' + ', '.join(TOOLS)
        else:
            try: observation = TOOLS[tool](**args)
            except Exception as e: observation = f'Tool error: {e}'
        # OBSERVE: add what happened to the record, print a short version, and give it to the AI.
        seen += f' {args} {observation}'
        if verbose:
            short = observation if len(observation) < 300 else observation[:300] + ' ...'
            print(f'        action: {tool}({args})')
            print(f'        result: {short}\n')
        messages.append({'role': 'user', 'content': f'Observation:\n{observation}'})
    # If we get here, the AI used up all its steps without a final answer.
    print('Stopped: reached the step limit without a final answer.')
    return None

# Run the agent on a first question. Watch the trace: which tool does it pick first, and why?
result = run_agent('Which email accounts are signed in on this device?')

## Part B: Why the Agent Matters

Here you ask the **same question two ways**: once to the AI model on its own (no tools, no
access to the phone image), and once through your agent. The model on its own has never seen this
phone, so whatever it says can only be a guess. Comparing the two replies shows why, in a forensic
setting, an answer is only useful when it points back to the evidence it came from.

### Step B.1: Ask the AI model with no tools

**Purpose:** see what an AI model says about this phone when it cannot look at any evidence.

**Input:** only the connection `client` and `MODEL` from Step 0.2. No tools, no rulebook, and no
image: just the question.

**How it works:** the cell sends the question straight to the model, one time, and prints the
reply. Nothing is checked.

**Output:** the model's reply, saved as `bare`. Expect it to say it cannot know, or to give a
general or invented answer. Keep it in mind for Step B.2 and your write-up.

In [ ]:
# This cell asks the AI model the question directly, with no tools and no access to the image.
bare = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Which email accounts are signed in on this device? Answer directly.'}],
).choices[0].message.content
# Show the reply. Ask yourself: what evidence is this based on?
print(bare)

### Step B.2: Ask the same question through the agent

**Purpose:** get an answer to the same question that is backed by a file from the phone image.

**Input:** the same question as Step B.1, and the `run_agent` loop you built in Step A.3 (with its
tools and rulebook).

**How it works:** exactly as in Step A.3: the agent opens files with its tools, and the evidence
check makes sure it cites a file it really saw before its answer is accepted.

**Output:** the step-by-step trace, then `ANSWER:` and `EVIDENCE:`. Put it next to the reply from
Step B.1: which one could you defend in a report, and why?

In [ ]:
# This cell asks the same question again, this time through the agent and its tools.
result = run_agent('Which email accounts are signed in on this device?')

## Part C: Ask Your Own Investigation Questions

Now you are the examiner. You use `run_agent(...)` to ask the phone image your own questions, and
you check each answer against the file it cites.

### Step C.1: Ask about the phone itself

**Purpose:** practise with a question that needs a different artifact than Part B did.

**Input:** the `run_agent` loop from Step A.3 and the phone image from Step 0.3.

**How it works:** the agent has to work out on its own that the phone's model and Android version
are kept in a system file at the top of the image (not in an app database), find it, and open it.

**Output:** the trace and a cited answer. Look at which file it cites: does that file really say
what the answer claims? Your own questions in Step C.2 work the same way.

In [ ]:
# This cell asks the agent a new question. To ask your own, change the text in quotes and re-run.
result = run_agent('What phone model is this and which Android version does it run?')

### Step C.2: Good questions to try

Change the question in the Step C.1 cell (or add a new cell with `run_agent('your question')`)
and run it. For each answer, look at the trace and check the cited file yourself.

- *How many contacts are saved on this device?*
- *What is the most recent text message that was sent, and to which number?*
- *Which messaging or social apps are installed?*  (hint: the agent can `search_text` or read `packages.list`)
- *What Wi-Fi networks has this device connected to?*
- *Find any file that mentions WhatsApp, then tell me the account's push name.*

If the agent takes a wrong turn or runs out of steps, re-run the cell, or name the kind of artifact
you expect (an account, a message, an app, a Wi-Fi network). Small free models make mistakes;
that is exactly why the answer must cite a file you can check.

## Optional: Run Sectra on a REAL Extraction

The image above is made up, so the lab runs instantly for everyone. But the agent explores
whatever folder you point it at, so you can aim it at a **real** phone extraction too.

**One honest warning:** a real *full file system* (FFS) extraction (a copy of every file on the
phone) is big. The public, no-login teaching image below (a Google Pixel 3 running Android 13,
published by Josh Hickman and mirrored by **Digital Corpora**) is about **18.5 GB to download and
about 40 GB once unpacked**. That is fine on a local machine or a Colab session with plenty of disk,
but it is a real download, so the cell that fetches it is **commented out** (switched off with `#`)
and nothing downloads until you choose to. Your three options:

1. **Synthetic** (default): the practice image you already used; instant.
2. **Real image:** switch on the fetch in Step R.2 (big; best run locally or with plenty of disk).
3. **Bring your own:** already have an extraction folder? Run `use_extraction('/path/to/it')`.

Other good public sources: Digital Corpora's other mobile images (`digitalcorpora.org`), and Josh
Hickman's image index at `thebinaryhick.blog/public_images`.

### Step R.1: Get the download helpers ready

**Purpose:** prepare two helpers: one that downloads and unpacks the real image safely, and one
that points the agent at a different evidence folder.

**Input:** nothing new. The helpers use `ROOT` and the tools from Part A.

**How it works:** the cell only *defines* the helpers; it downloads nothing.

- `fetch_extraction(...)` first checks that you have enough free disk space, then downloads the
  image, then unpacks it. While unpacking it skips **links** (files that point to other places), so
  nothing in the download can write outside its own folder.
- `use_extraction(path)` changes `ROOT` to the folder you name. The tools from Step A.1 always look
  inside `ROOT`, so from then on every `run_agent(...)` question searches that folder instead.

**Output:** one line, `Ready. ...`. Step R.2 is where you choose to use these helpers.

In [ ]:
# This cell defines helpers for using a REAL extraction. It does not download anything by itself.
import shutil, tarfile, urllib.request

# The list of real images we know how to fetch: a dictionary of name -> details.
REAL_SOURCES = {
    # Josh Hickman's Pixel 3 / Android 13 FFS image, mirrored by Digital Corpora
    # (a public academic archive, no login needed).
    'digitalcorpora_android13': {
        'url': 'https://digitalcorpora.s3.amazonaws.com/corpora/mobile/android_13/android_13_data.tar.gz',
        'download_gb': 18.5, 'extracted_gb': 40,
    },
}

# Download one file and print progress, so you can see a long download is still moving.
def _download(url, dest):
    last = [0]
    def hook(block, bs, total):
        done = block * bs
        if total > 0 and done - last[0] > 250 * 1024 * 1024:   # report every ~250 MB
            last[0] = done; print(f'  {done/1e9:5.1f} / {total/1e9:.1f} GB  ({100*done/total:3.0f}%)')
    urllib.request.urlretrieve(url, dest, reporthook=hook)

# Download AND unpack a real image, with safety checks first.
def fetch_extraction(source='digitalcorpora_android13', dest='real_extraction', max_gb=60, force=False):
    """Download + extract a real public FFS image. Large - read the printed size first."""
    # 1. Check the disk: stop early if there is not enough free space (unless force=True).
    s = REAL_SOURCES[source]
    need = s['download_gb'] + s['extracted_gb']
    free = shutil.disk_usage('.').free / 1e9
    print(f"{source}: ~{s['download_gb']} GB download, ~{s['extracted_gb']} GB extracted.")
    print(f'Free disk here: {free:.0f} GB.  Needed (rough): {need:.0f} GB.')
    if not force and free < need:
        raise SystemExit('Not enough free disk. Use the synthetic image, bring your own, or run locally. '
                         '(force=True to try anyway.)')
    if not force and s['download_gb'] > max_gb:
        raise SystemExit(f"This image is ~{s['download_gb']} GB (> max_gb={max_gb}). Raise max_gb or force=True.")
    # 2. Download (skipped if a finished download is already there).
    dest = Path(dest); dest.mkdir(exist_ok=True)
    tgz = dest / 'image.tar.gz'
    if not tgz.exists():   # download to a .part file first, so an interrupted download is never reused
        print('Downloading (this takes a while)...'); _download(s['url'], tgz.with_suffix('.part'))
        tgz.with_suffix('.part').rename(tgz)
    # 3. Unpack, skipping links, so nothing in the archive can land outside the folder.
    print('Extracting (links are skipped, so nothing can point outside the folder)...')
    with tarfile.open(tgz) as t:
        if hasattr(tarfile, 'data_filter'):   # Python 3.12+: the safe 'data' filter, minus links
            t.extractall(dest, filter=lambda m, p: None if (m.issym() or m.islnk()) else tarfile.data_filter(m, p))
        else:
            # Older Python: pick the safe entries ourselves (no links, no paths that climb out).
            t.extractall(dest, members=[m for m in t.getmembers() if not (m.issym() or m.islnk())
                                        and not m.name.startswith('/') and '..' not in m.name.split('/')])
    print('Done. Extracted under', dest.resolve())
    return dest

# Point the agent at a different evidence folder by changing ROOT, which every tool uses.
def use_extraction(path='real_extraction'):
    """Point Sectra at an extraction folder: rebinds ROOT, so your run_agent(...) cells now walk it."""
    global ROOT
    ROOT = Path(path)
    print('ROOT is now', ROOT.resolve())
    try:
        print('Top-level entries:', sorted(os.listdir(ROOT))[:20])
    except Exception as e:
        print('(could not list it yet:', e, ')')
    print('Re-run your run_agent(...) cells - Sectra now investigates this extraction.')
    return ROOT

print('Ready. Uncomment the next cell to fetch a real image, or call use_extraction(\'/your/path\').')

### Step R.2: Fetch and use the real image (switched off until you choose)

**Purpose:** run the agent on real evidence instead of the practice image.

**Input:** the helpers from Step R.1, and plenty of free disk space.

**How it works:** every line below starts with `#`, so running the cell does nothing. To use it,
remove the `#` at the start of each line, then run it: it fetches the image, points `ROOT` at it,
and asks one question. On a real extraction, prefer `list_dir`, `find_files`, and `run_sql` over
`search_text` (which reads every text file and can be slow), and ask targeted questions.

**Output:** nothing while the lines are switched off. When switched on: download progress, the
top folders of the real image, and then the agent's trace and cited answer.

In [ ]:
# root = fetch_extraction('digitalcorpora_android13')   # ~18.5 GB download - only when you mean it
# use_extraction(root)
# run_agent('Which messaging apps are installed, and what is the device model?')

## Part D (Optional, Advanced): Let the Agent Write Code

### Step D.1: A tool that runs Python the AI writes

**Purpose:** see how an agent could handle questions the five fixed tools do not cover, by writing
its own small piece of Python, and why that is risky.

**Input:** `ROOT` (the image folder) and the `TOOLS` toolbox from Step A.1.

**How it works:** the cell writes a sixth tool, `run_python`, that runs a short piece of Python and
returns what it printed. This is powerful but risky: the AI's text becomes code that runs on your
machine. A keyword check blocks obvious dangerous words (such as `open(` or `subprocess`), but it
only stops accidents, **not** someone determined: this is **not a sandbox** (a sealed-off space
where code cannot harm anything). The new tool is **not** given to the agent unless you remove the
`#` from the `TOOLS['run_python']` line, and you would also need to add `run_python` to the tool list
in the rulebook from Step A.2. Use it only on the practice image or a copy of an extraction, never
on original evidence.

**Output:** the cell calls the tool once itself, as a test, and prints the names of the database
files in the image.

In [ ]:
# ADVANCED, OPTIONAL: this cell writes a tool that runs a short piece of Python the AI writes.
import io, contextlib

def run_python(code):
    # The snippet can use ROOT (the image folder). The keyword check below blocks obvious
    # dangerous words. It stops accidents; it is NOT a sandbox.
    if any(b in code for b in ('open(', 'write', 'unlink', 'remove', 'rmtree', 'os.system',
                               'subprocess', 'Popen', '__import__')):
        return 'Blocked: that snippet looks like it writes or runs external commands.'
    # Run the snippet and catch whatever it prints, so we can hand that text back to the AI.
    buf = io.StringIO(); safe = {'ROOT': ROOT, 'sqlite3': sqlite3, 'json': json, 'Path': Path, 're': re}
    try:
        with contextlib.redirect_stdout(buf): exec(code, safe)
    except Exception as e:
        return f'Python error: {e}'
    out = buf.getvalue().strip()
    return out[:1500] if out else '(ran, but printed nothing)'

# To enable it, uncomment the next line (and mention run_python in the system prompt):
# TOOLS['run_python'] = run_python
print(run_python("print(sorted(p.name for p in ROOT.rglob('*.db')))"))   # a direct test call

## Your Short Write-Up

Answer these three questions in your own words:

1. Pick one question. Paste the agent's answer **and** the evidence path, then check the file.
2. Compare Part B.1 and B.2: what did the plain model do that the agent did not?
3. In one sentence: why does making the agent **cite the file** make its answer safer to rely on?

**How to use the cell below:** replace each `'...'` with your answer (keep the quote marks), then
run the cell. It prints your three answers so they are saved in the notebook.

In [ ]:
# Type your answers between the quote marks, then run this cell to print them.
writeup = {
    'q1_answer_and_evidence': '...',
    'q2_plain_vs_agent': '...',
    'q3_why_citing_helps': '...',
}
for k, v in writeup.items(): print(k, '->', v)

## Submit

Save the notebook with the reasoning trace, the Part B comparison, and your write-up.

---

*The device image is synthetic. The agent (tools + the think-act-observe loop) is built from scratch, driven by a small open-source Qwen model running locally via Ollama, so you can see every step and the exact file behind every answer.*